# LoRA / VeRA / LoRA-XS / TinyLoRA GLUE Colab Runner

这个 notebook 是最终 Colab 运行入口。它使用 Hugging Face `datasets` 从公开 GLUE 数据集下载真实数据，不使用 `smoke_test.py` 里的假数据。

In [ ]:
!pip -q install -U transformers datasets evaluate accelerate scikit-learn pandas

## 1. 指向项目目录

如果你把整个项目上传到 Colab，确认 `PROJECT_DIR` 指向包含 `comparsion and evaluation experinment` 的目录。

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path('/content/lora_vera_loraxs')
if (Path.cwd() / 'comparsion and evaluation experinment').exists():
    PROJECT_DIR = Path.cwd()

EXPERIMENT_DIR = PROJECT_DIR / 'comparsion and evaluation experinment'
assert EXPERIMENT_DIR.exists(), f'Cannot find experiment dir: {EXPERIMENT_DIR}'
sys.path.insert(0, str(EXPERIMENT_DIR))
print('EXPERIMENT_DIR =', EXPERIMENT_DIR)

## 2. 查看可运行的 model settings

`model_settings/*.py` 是 model 粒度；`experiment_settings/*.py` 是 experiment 粒度。

In [ ]:
from model_settings import ALL_MODEL_SETTINGS
import pandas as pd

pd.DataFrame([
    {
        'name': s.name,
        'method': s.method,
        'rank': s.rank,
        'targets': ','.join(s.target_modules),
        'basis': s.basis,
        'tiny_u': s.tiny_projection_dim,
        'tiny_tie_every': s.tiny_tie_every,
        'tags': ','.join(s.tags),
    }
    for s in ALL_MODEL_SETTINGS
])

## 3. 配置一次真实 GLUE 训练

默认配置是 Colab 快速验证：SST-2、RoBERTa-base、LoRA q/v rank 8，只取部分样本。要跑完整实验，把 `MAX_TRAIN_SAMPLES = None`、`MAX_EVAL_SAMPLES = None`，并增加 epoch。

In [ ]:
from experiments.colab_glue_runner import GlueRunConfig, run_glue_experiment

TASK = 'sst2'  # 可选: sst2, mrpc, cola, qnli, rte, stsb, mnli, qqp
BASE_MODEL = 'roberta-base'
MODEL_SETTING = 'lora_qv_r8'  # 例如: vera_qv_r256, loraxs_qv_r8, loraxs_qvo_fc1_r16

MAX_TRAIN_SAMPLES = 2000   # None 表示完整 train split
MAX_EVAL_SAMPLES = 1000    # None 表示完整 validation split

config = GlueRunConfig(
    task=TASK,
    model_setting_name=MODEL_SETTING,
    base_model_name=BASE_MODEL,
    output_dir=str(EXPERIMENT_DIR / 'output' / 'colab_glue'),
    seed=1,
    max_length=128,
    learning_rate=2e-4,
    num_train_epochs=1.0,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    max_train_samples=MAX_TRAIN_SAMPLES,
    max_eval_samples=MAX_EVAL_SAMPLES,
    train_classifier_head=True,
    fp16=True,
)
config

## 4. 运行训练

这一格会下载公开 GLUE 数据集和 Hugging Face 模型权重，并进行真实 fine-tuning。

In [ ]:
summary = run_glue_experiment(config)
summary['eval_metrics']

In [ ]:
important = {
    'run_id': summary['run_id'],
    'task': summary['task'],
    'method': summary['model_setting']['method'],
    'setting': summary['model_setting']['name'],
    'trainable_params': summary['trainable_params'],
    'trainable_ratio': summary['trainable_ratio'],
    'classifier_trainable_params': summary['classifier_trainable_params'],
    'svd_time_seconds': summary['svd_time_seconds'],
    'elapsed_seconds': summary['elapsed_seconds'],
    'peak_memory_gb': summary['peak_memory_bytes'] / 1024**3,
    **summary['eval_metrics'],
}
pd.DataFrame([important])

## 5. 批量比较多个 settings

Colab 快速跑建议先用小样本；正式结果再切到完整数据。

In [ ]:
SETTINGS_TO_RUN = [
    'lora_qv_r8',
    'vera_qv_r256',
    'loraxs_qv_r8',
]

rows = []
for setting_name in SETTINGS_TO_RUN:
    cfg = GlueRunConfig(
        task=TASK,
        model_setting_name=setting_name,
        base_model_name=BASE_MODEL,
        output_dir=str(EXPERIMENT_DIR / 'output' / 'colab_glue'),
        seed=1,
        max_train_samples=MAX_TRAIN_SAMPLES,
        max_eval_samples=MAX_EVAL_SAMPLES,
        num_train_epochs=1.0,
        learning_rate=2e-4,
        train_classifier_head=True,
        fp16=True,
    )
    result = run_glue_experiment(cfg)
    rows.append({
        'setting': setting_name,
        'method': result['model_setting']['method'],
        'trainable_params': result['trainable_params'],
        'svd_time_seconds': result['svd_time_seconds'],
        'elapsed_seconds': result['elapsed_seconds'],
        **result['eval_metrics'],
    })

results_df = pd.DataFrame(rows)
results_df